# pgvector(PostgreSQL 확장)

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()

POSTGRESQL_PORT = os.getenv("POSTGRESQL_PORT")
POSTGRESQL_USER = os.getenv("POSTGRESQL_USER")
POSTGRESQL_PASSWORD = os.getenv("POSTGRESQL_PASSWORD")
POSTGRESQL_DBNAME = os.getenv("POSTGRESQL_DBNAME")


In [ ]:
# postgreSQL 접속 테스트
conn = psycopg2.connect(
    host="localhost",   # 클라우드였다면 VM 공인 IP
    port=POSTGRESQL_PORT,
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    dbname=POSTGRESQL_DBNAME
)

# Table 설계
- id : primary key용
- qa_id : 파일명 (qa_data_qa_id 값)

- instruction : 요구사항 (qa_data_instruction)
- question : 고객질문 (qa_data_input_question) 
- answer : 상담사답변 (qa_data_input_answer) 
- follow_up_question : 꼬리질문 (qa_data_input_follow_up_question)
- output : 종합답변 (qa_data_output)

- full_source : “요구사항:~\n고객질문:~\n상담사답변:~\n꼬리질문:~\n종합답변:~”꼴로 이루어진 전처리값

- consulting_category
- consulting_topic
- qa_topic
- consulting_purpose

- embedding_q : question에 대한 임베딩 값
- embedding_i : instruction에 대한 임베딩 값
- embedding_full : full_source에 대한 임베딩 값


In [ ]:
'''
qa_data_qa_id: 결측치 0건
qa_data_instruction: 결측치 0건
qa_data_input_question: 결측치 0건
qa_data_input_answer: 결측치 0건
qa_data_input_follow_up_question: 결측치 0건
qa_data_output: 결측치 0건
consulting_consulting_category: 결측치 0건
'''

# Embedding을 위한 GPU 연결 
#### (기존 커널에서 colab 커널로 변경)

- VSCode에 Google Colab 확장 설치 필요
- 커널 Colab > New Colab Server > GPU > T4 선택
- 8만건 임베딩을 진행해야 하기 때문에 GPU 활용 권장
- Colab 무료 티어의 경우 세션 시간 제한이 있어 분할 또는 중간 저장 로직 추가 필요

In [ ]:
# colab GPU(T4) 연결 확인
!nvidia-smi 

In [ ]:
# colab GPU(T4) 연결 확인 2
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0))
print("PyTorch 버전:", torch.__version__)

In [ ]:
# 작업 공간 확인용
# 현재의 작업 디렉토리와 맞지 않고 아래와 같이 출력된다면 Colab 원격 서버 임시 작업 공간인 것
# /content
# ['.config', 'sample_data']
# 이 경우, Colab 원격 서버로 데이터 옮겨주어야 함
import os

print(os.getcwd())
print(os.listdir("."))

In [ ]:
# Colab 원격 서버에 구글 드라이브 파일 다운
# * 구글 드라이브 파일 install 위해 "gdown" 라이브러리 추가 설치
# 민감한 데이터는 다운 후 드라이브에서 삭제하기
import gdown

#https://docs.google.com/spreadsheets/d/1IqNn72Fftb4KjHSZbPOrmeFWva7wLOs-/edit?usp=sharing&ouid=106711546381992123673&rtpof=true&sd=true

file_id = "1IqNn72Fftb4KjHSZbPOrmeFWva7wLOs-"
output = "./dontalk/data/data.xlsx"

gdown.download(id=file_id, output=output, quiet=False)

In [ ]:
# 확인용
import os
print(os.listdir("./dontalk/data/"))

In [ ]:
# df 확인
import pandas as pd

file_path = "./dontalk/data/data.xlsx"
df = pd.read_excel(file_path)
df

In [ ]:
# 필요한 컬럼만 추출 + 스키마에 맞게 이름 변경
column_map = {
    "qa_data_qa_id": "qa_id",
    "qa_data_instruction": "instruction",
    "qa_data_input_question": "question",
    "qa_data_input_answer": "answer",
    "qa_data_input_follow_up_question": "follow_up_question",
    "qa_data_output": "output",
    "qa_data_full_source": "full_source",
    "consulting_consulting_category": "consulting_category",
    "consulting_consulting_topic": "consulting_topic",
    "qa_data_qa_topic": "qa_topic",
    "qa_data_consulting_purpose": "consulting_purpose",
}

df_selected = df[list(column_map.keys())].rename(columns=column_map).copy()
print(f"전체 행 수: {len(df_selected)}")
df_selected.head(3)

In [ ]:
# 결측치 확인용
required_cols = ["qa_id", "instruction", "question", "consulting_category"]
for col in required_cols:
    missing = df_selected[col].isna().sum()
    print(f"{col}: 결측치 {missing}건")

# 결측이 있다면 해당 행 제외
before = len(df_selected)
df_selected = df_selected.dropna(subset=required_cols)
after = len(df_selected)
print(f"\n제외된 행: {before - after}건, 최종 사용 행: {after}건")

### Embedding Test
- 코드 에러 확인 및 시간 체크용 100건 샘플 테스트

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

# 임베딩 모델 로드
model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko", device=device)
print("모델 로드 완료")

In [ ]:
import time

df_sample = df_selected.head(100).copy()

def encode_texts(texts, label):
    if texts == df_sample["full_source"].tolist():
        b_size = 16
    else:
        b_size = 32

    start_time = time.time()
    embeddings = model.encode(
        texts, 
        batch_size=b_size, 
        show_progress_bar=True,
        normalize_embeddings=True,   # 코사인 유사도 검색을 위한 정규화
        convert_to_numpy=True,)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"[{label}] 임베딩 완료. shape={embeddings.shape}\n소요 시간: {elapsed_time:.2f}초")
    return embeddings

start = time.time()

emb_q_sample = encode_texts(df_sample["question"].tolist(), "embedding_q")
emb_i_sample = encode_texts(df_sample["instruction"].tolist(), "embedding_i")
emb_full_sample = encode_texts(df_sample["full_source"].tolist(), "embedding_full")

elapsed = time.time() - start


print(f"\n전체 임베딩 소요 시간: {elapsed:.2f}초")
print(f"8만 건 예상 소요 시간(단순 비례): 약 {elapsed * 800 / 60:.1f}분")

print(len(emb_q_sample))


In [ ]:
len(emb_q_sample)